# B2-rep — Kaggle Test Census Execution (Runtime Locked)

**Giao thức Đăng ký trước (OSF Frozen Protocol) — Bước 5/ Bước 7 Confirmatory Test:**
- **Arm:** `B2-rep`
- **Tập dữ liệu:** Toàn bộ census Test (Devign 2,732 + ReVeal 2,274).
- **Runtime Lock:** Tuân thủ 100% hợp đồng `grace_qwen_q4_runtime_v1` (Dũng 2026-11-07):
  * `split_mode: LLAMA_SPLIT_MODE_LAYER` trên 2x Tesla T4 (Kaggle Linux)
  * Assert đúng 49/49 layers nạp trọn vẹn vào GPU
  * Exact packages: `numpy==2.1.3`, `huggingface-hub==0.36.2`, `tokenizers==0.22.2`, `transformers==4.57.1`, `scikit-learn==1.5.2`, `llama-cpp-python==0.3.16` (SHA-256 verified)
  * Đánh giá trực tiếp tại ngưỡng khóa cố định $\tau^*$ từ validation (không refit).


**Ghi chú bắt buộc (AGENTS.md 2.6):** Arm B2-rep bắt buộc ghi `repeat: true` và `tag: 'repeat'`. Cách ly cache triệt để.

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, traceback, shutil, platform, importlib.metadata, zipfile, hashlib, time, re
from datetime import datetime, timezone

SESSION_STARTED = time.time()
DIAG = Path('/kaggle/working/grace_diagnostics')
DIAG.mkdir(parents=True, exist_ok=True)
EVENTS = DIAG / 'events.log'

def note(message):
    text = str(message)
    print(text, flush=True)
    with EVENTS.open('a', encoding='utf-8') as f:
        f.write(f'[{datetime.now(timezone.utc).isoformat()}] {text}\n')

def diagnostics_zip():
    archive = shutil.make_archive('/kaggle/working/grace_diagnostics', 'zip', root_dir=DIAG)
    print('Tải log chẩn đoán từ Kaggle Output:', archive, flush=True)
    return archive

def run_logged(cmd, name, env=None):
    log = DIAG / f'{name}.log'
    note(f'Bắt đầu {name}: {cmd!r} | log đầy đủ: {log}')
    is_inference = name.startswith('resume_seed_')
    with log.open('w', encoding='utf-8') as f:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, encoding='utf-8', errors='replace', bufsize=1, env=env)
        for line in proc.stdout:
            f.write(line); f.flush()
            if re.search(r'\b(?:Found|Skip(?:ping)?) link\b', line, flags=re.IGNORECASE):
                continue
            if is_inference:
                match = re.search(r'^(\S+).*\((\d+)/(\d+)\)\s*$', line.strip())
                if match:
                    run_key, done, total = match.group(1), int(match.group(2)), int(match.group(3))
                    if done % 100 == 0 or done == total:
                        print(f'Inference {run_key}: {done}/{total}', flush=True)
                elif line.strip():
                    print(line, end='', flush=True)
            else:
                print(line, end='', flush=True)
        code = proc.wait()
    if is_inference and '--status' in cmd:
        status_path = Path(cmd[cmd.index('--status') + 1])
        if status_path.is_file():
            status = json.loads(status_path.read_text(encoding='utf-8'))
            print(f"Inference summary: {status.get('completed_records_total', 0)} records; state={status.get('state')}", flush=True)
            for run_key, progress in status.get('runs', {}).items():
                print(f"  {run_key}: {progress.get('completed', 0)}/{progress.get('count', '?')} [{progress.get('state', 'pending')}]")
    note(f'{name} kết thúc với mã {code}')
    if code:
        raise subprocess.CalledProcessError(code, cmd)

def versions():
    result = {}
    for name in ('llama-cpp-python', 'transformers', 'huggingface_hub', 'scikit-learn', 'numpy', 'scipy'):
        try: result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError: result[name] = 'not installed'
    note('Installed package versions: ' + json.dumps(result))
    return result

def capture_failure(shell, etype, evalue, tb, tb_offset=None):
    trace = ''.join(traceback.format_exception(etype, evalue, tb))
    try:
        with (DIAG / 'exceptions.log').open('a', encoding='utf-8') as f:
            f.write(f'\n[{datetime.now(timezone.utc).isoformat()}]\n{trace}\n')
        note(f'Lỗi: {etype.__name__}: {evalue}')
        versions()
        diagnostics_zip()
    except Exception as diagnostic_error:
        print('Không thể ghi đầy đủ chẩn đoán:', diagnostic_error, flush=True)
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
    return []

get_ipython().set_custom_exc((Exception,), capture_failure)
note('UTC: ' + datetime.now(timezone.utc).isoformat())
note('Python: ' + sys.version)
note('Executable: ' + sys.executable)
note('Platform: ' + platform.platform())
for loc in ('/kaggle/working', '/kaggle/input'):
    try:
        usage = shutil.disk_usage(loc)
        note(f'Disk {loc}: total={usage.total} free={usage.free} bytes')
    except OSError as exc: note(f'Disk {loc}: {exc}')
versions()
for name, cmd in [('gpu', ['nvidia-smi']), ('cuda_compiler', ['nvcc', '--version']), ('cmake', ['cmake', '--version']), ('pip', [sys.executable, '-m', 'pip', '--version'])]:
    try: run_logged(cmd, name)
    except (OSError, subprocess.CalledProcessError) as exc: note(f'{name}: {type(exc).__name__}: {exc}')
try:
    input_entries = [p.name for p in Path('/kaggle/input').glob('*')]
    note(f'Input entries: {input_entries}')
    gguf_files = [str(p) for p in Path('/kaggle/input').rglob('*.gguf')]
    note(f'Input GGUF files: {gguf_files}')
except Exception as exc: note(f'Input inspection error: {exc}')
diagnostics_zip()
RUNTIME_LOCK_BYTES = b'{\n  "schema_version": 1,\n  "contract_id": "grace_qwen_q4_runtime_v1",\n  "created_on": "2026-11-07",\n  "timezone": "Asia/Bangkok",\n  "status": "core_policy_locked_notebook_adoption_pending",\n  "scope": "New GRACE diagnostic Qwen Q4 inference on Kaggle only; excludes Colab/Vast and preparation/CodeT5/legacy API prototypes",\n  "user_decisions": {\n    "inference_seed": 4294967295,\n    "packages": "exact versions approved",\n    "gpu_policy": "layer + auto free VRAM, approved",\n    "cpu_threads": "approved: preserve default CPU-count formulas used by all 16 notebooks",\n    "build": "approved: Kaggle platform build with mandatory complete provenance; no shared container required"\n  },\n  "model": {\n    "repo": "ggml-org/Qwen3-Coder-30B-A3B-Instruct-GGUF",\n    "filename": "Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf",\n    "sha256": "f6050b4da22869aa3fb9d94bb3314e9fc5a027d4741c1f97761c387d6b2b2335",\n    "quantization": "Q4_K_M",\n    "verify_sha256_before_load": true,\n    "adapters_allowed": false\n  },\n  "tokenizer": {\n    "repo": "Qwen/Qwen3-Coder-30B-A3B-Instruct",\n    "revision": "b2cff646eb4bb1d68355c01b18ae02e7cf42d120",\n    "chat_template": {\n      "role": "user",\n      "add_generation_prompt": true\n    },\n    "gguf_tokenize": {\n      "add_bos": false,\n      "special": true\n    },\n    "require_hf_gguf_token_id_equality": true,\n    "require_single_token_binary_labels": true\n  },\n  "packages": {\n    "numpy": "2.1.3",\n    "huggingface-hub": "0.36.2",\n    "tokenizers": "0.22.2",\n    "transformers": "4.57.1",\n    "llama-cpp-python": "0.3.16",\n    "scikit-learn": "1.5.2"\n  },\n  "backend_source": {\n    "name": "llama-cpp-python",\n    "version": "0.3.16",\n    "sdist_filename": "llama_cpp_python-0.3.16.tar.gz",\n    "sdist_sha256": "34ed0f9bd9431af045bb63d9324ae620ad0536653740e9bb163a2e1fcb973be6",\n    "verify_sdist_before_build": true,\n    "allow_other_backend": false,\n    "record_runner_source_sha256": true\n  },\n  "constructor": {\n    "n_ctx": 8192,\n    "n_batch": 256,\n    "n_ubatch": 256,\n    "n_gpu_layers": -1,\n    "split_mode": "LLAMA_SPLIT_MODE_LAYER",\n    "main_gpu": 0,\n    "tensor_split": null,\n    "seed": 4294967295,\n    "logits_all": true,\n    "flash_attn": false,\n    "offload_kqv": true,\n    "use_mmap": true,\n    "use_mlock": false,\n    "embedding": false\n  },\n  "gpu": {\n    "profile": "kaggle_t4_x2",\n    "device_count": 2,\n    "device_model": "NVIDIA Tesla T4",\n    "distribution": "layer_by_free_vram_at_load",\n    "random_distribution": false,\n    "fixed_ratio_guaranteed": false,\n    "record_free_and_total_vram": true,\n    "record_device_order_and_uuid": true,\n    "record_layer_and_buffer_offload_report": true,\n    "required_offloaded_layers": 49,\n    "allow_silent_cpu_layer_fallback": false,\n    "allow_auxiliary_host_buffers": true,\n    "other_hardware_requires_explicit_profile_review": true\n  },\n  "threads": {\n    "policy": "preserve_backend_default_formulas",\n    "n_threads": "max(multiprocessing.cpu_count() // 2, 1)",\n    "n_threads_batch": "multiprocessing.cpu_count()",\n    "fixed_thread_count": null,\n    "record_effective_counts": true,\n    "approval_status": "user_approved"\n  },\n  "build": {\n    "GGML_CUDA": true,\n    "CMAKE_CUDA_ARCHITECTURES": "75",\n    "CMAKE_LIBRARY_PATH": "/usr/local/nvidia/lib64",\n    "CMAKE_BUILD_PARALLEL_LEVEL": "2",\n    "FORCE_CMAKE": "1",\n    "python_family": "3.13",\n    "exact_python_patch": null,\n    "exact_cuda_toolkit": null,\n    "exact_host_compiler": null,\n    "exact_cmake": null,\n    "compiled_wheel_sha256": null,\n    "binary_environment_fully_locked": false,\n    "policy": "Kaggle platform build is approved. Record exact Python patch, CUDA/nvcc/driver, compiler, CMake, flags, dependencies and compiled backend hash. A shared container or bit-identical binary is not required or claimed."\n  },\n  "scoring": {\n    "name": "unconstrained_first_token_logits_v1",\n    "reset_before_each_prompt": true,\n    "evaluate": "llm.eval(gguf_ids)",\n    "logits": "llm.scores[llm.n_tokens - 1]",\n    "normalization_dtype": "float64",\n    "logprob": "stable full-vocabulary log_softmax of last-position logits",\n    "s": "exp(logprob_1 - logaddexp(logprob_0, logprob_1))",\n    "pred_argmax": "int(s >= 0.5)",\n    "raw_token": "argmax of full-vocabulary logits",\n    "reject_non_finite": true,\n    "retain_valid_score_if_raw_not_binary": true\n  },\n  "decoding": {\n    "mode": "direct_logits_and_greedy_argmax",\n    "sampling": false,\n    "generation_api": null,\n    "temperature": null,\n    "top_p": null,\n    "top_k": null,\n    "note": "Sampling parameters are not used by eval; the default-seed sentinel does not promise fixed RNG output for future sampling paths."\n  },\n  "prompt_contract": {\n    "prompt_selection_seeds": [\n      42,\n      43,\n      44\n    ],\n    "selection_seed_distinct_from_inference_seed": true,\n    "identity": [\n      "arm",\n      "prompt seed",\n      "dataset",\n      "split",\n      "ID",\n      "bundle SHA256",\n      "prompt SHA256"\n    ],\n    "verify_frozen_input_and_each_prompt_hash": true,\n    "silent_reformatting_or_truncation": false,\n    "different_arms_use_their_own_frozen_prompts": true\n  },\n  "thresholds": {\n    "source": "existing frozen arm-specific threshold contracts",\n    "test_refit_allowed": false,\n    "preserve_existing_probe_inheritance": true\n  },\n  "resume": {\n    "require_match": [\n      "arm",\n      "prompt seed",\n      "dataset",\n      "split",\n      "ID",\n      "label",\n      "prompt SHA256",\n      "model SHA256",\n      "tokenizer revision",\n      "backend source",\n      "scoring",\n      "constructor",\n      "threshold lock when applicable"\n    ],\n    "changed_contract": "do not silently append to historical results; use separate run identity or reviewed migration",\n    "rewrite_historical_outputs": false\n  },\n  "provenance_required": [\n    "contract SHA256",\n    "all package versions including transitive dependencies",\n    "exact Python/OS/libc",\n    "CUDA toolkit/nvcc/driver",\n    "GPU device order/UUID/VRAM",\n    "host compiler/CMake/build flags",\n    "source archive SHA256",\n    "compiled backend/wheel SHA256",\n    "runner source SHA256",\n    "effective CPU threads",\n    "effective batch/microbatch",\n    "offload log",\n    "model/tokenizer/prompt/threshold identities"\n  ],\n  "integration": {\n    "existing_notebooks_auto_enforce_json": false,\n    "required_before_claiming_compliance": "load, validate and apply this JSON; abort on incompatible values",\n    "historical_runs_are_not_retroactively_certified": true\n  },\n  "open_questions": [\n    {\n      "id": "missing_run_logs",\n      "arms": [\n        "B1-full",\n        "B1-shuf",\n        "B2-rep",\n        "B2-nolabel",\n        "B2-labelonly"\n      ],\n      "status": "awaiting executed notebooks/install logs"\n    },\n    {\n      "id": "b3_runners",\n      "arms": [\n        "B3",\n        "B3-nodiff",\n        "B3-swap",\n        "B3-flip"\n      ],\n      "status": "no dedicated current Kaggle Qwen notebook found in GRACE-summary"\n    }\n  ],\n  "references": {\n    "package_hash": "https://pypi.org/pypi/llama-cpp-python/0.3.16/json",\n    "constructor_defaults": "https://raw.githubusercontent.com/abetlen/llama-cpp-python/v0.3.16/llama_cpp/llama.py",\n    "auto_split": "https://raw.githubusercontent.com/ggml-org/llama.cpp/4227c9be4268ac844921b90f31595f81236bd317/src/llama-model.cpp"\n  },\n  "notebook_audit": [\n    {\n      "path": "01_Author_Baseline_Unified/notebooks/02_kaggle_qwen_q4_b2.ipynb",\n      "file_sha256_at_audit": "d83c14d4f069e1987c024b27ac801840f7fa080d6fd3f75a17f2e004fdce9ba3",\n      "configured_inference_seed": 4294967295,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "02_Arm_B0_Source_Only/notebooks/02_kaggle_qwen_q4_b0.ipynb",\n      "file_sha256_at_audit": "fdc0ee82e6df6903463023471e2babef6fd6a136bfc4ce14e7dd8806376ee7d6",\n      "configured_inference_seed": 42,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "03_Arm_B1_Graph_Only/notebooks/02_kaggle_qwen_q4_b1.ipynb",\n      "file_sha256_at_audit": "c01dac2696b9391414d0f9c5ee0f7eb1f26e1ef0f69de13b953da83f75f18746",\n      "configured_inference_seed": 42,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "03_Arm_B1_Graph_Only/notebooks/02_kaggle_qwen_q4_b1_executed_I08.ipynb",\n      "file_sha256_at_audit": "6e501ab2f57b964e9c43a807297df4407620bcb9a02f0a3744d742341dce22fb",\n      "configured_inference_seed": 42,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "04_Arm_B2_Flip/notebooks/02_kaggle_qwen_q4_b2_flip.ipynb",\n      "file_sha256_at_audit": "c6e7ce9c1351ab88375cd6e4d9b7d34cd5bb9f7862108fef5d12d81196cf37fb",\n      "configured_inference_seed": 42,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "06_Arm_B2_Rand/notebooks/02_kaggle_qwen_q4_b2_rand.ipynb",\n      "file_sha256_at_audit": "b819048b5f126882dba6ff59581354937101195d8807974bc3a1f7cc65910c86",\n      "configured_inference_seed": 4294967295,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "06_Arm_B2_Rand/test_delivery_20261107/02_kaggle_qwen_q4_b2_rand_seed42_test.ipynb",\n      "file_sha256_at_audit": "a35b30da7d32ca935f0d5bba894ccb4fa02ba52e1212c70d6afded2ba08e3c7e",\n      "configured_inference_seed": 4294967295,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "06_Arm_B2_Rand/test_delivery_20261107/02_kaggle_qwen_q4_b2_rand_seed43_test.ipynb",\n      "file_sha256_at_audit": "42ebe992c4e716ced593efdd089a720d3d0019663b69f5faf303c201975264d1",\n      "configured_inference_seed": 4294967295,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "11_Arm_B1H2_Evidence/kaggle_b1h2_test_20261106/02_kaggle_qwen_q4_b1_h2_test_20261106.ipynb",\n      "file_sha256_at_audit": "3cb1f4d4fc2d4e9ca839d22db2b1660f671aebbd046bdd2a0e2e94f4a8541dc7",\n      "configured_inference_seed": 4294967295,\n      "tensor_split": [\n        1.0,\n        1.0\n      ],\n      "missing_explicit_package_pins": [],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "11_Arm_B1H2_Evidence/notebooks/02_kaggle_qwen_q4_b1_h2_validation_20261106.ipynb",\n      "file_sha256_at_audit": "57cdf5249395aec0f0c5bc47421a9d6fb3789fd5809de7d42bc80f632d91c88f",\n      "configured_inference_seed": 4294967295,\n      "tensor_split": [\n        1.0,\n        1.0\n      ],\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "Runs-management/kaggle_validation_6arms_20261104/02_kaggle_qwen_q4_b1_full_validation.ipynb",\n      "file_sha256_at_audit": "dcfff73568daddd1e97934c0f2175d4b36b36339032049fffb16160a541b34bf",\n      "configured_inference_seed": 4294967295,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "Runs-management/kaggle_validation_6arms_20261104/02_kaggle_qwen_q4_b1_shuf_validation.ipynb",\n      "file_sha256_at_audit": "d9ed03125e8f309e28b5f06d14be3038e3fe77277fe9ae0ac33857d777aa9759",\n      "configured_inference_seed": 4294967295,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "Runs-management/kaggle_validation_6arms_20261104/02_kaggle_qwen_q4_b2_labelonly_validation.ipynb",\n      "file_sha256_at_audit": "6a54f85e4537e66ab1a1c57615232e3b8a99850a70308c3d1bcadb0fcdb3d8db",\n      "configured_inference_seed": 4294967295,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "Runs-management/kaggle_validation_6arms_20261104/02_kaggle_qwen_q4_b2_nolabel_validation.ipynb",\n      "file_sha256_at_audit": "099325d9a24e2df72e43a33f77250ea1590302409d7d5cb1b5a7a035f605364a",\n      "configured_inference_seed": 4294967295,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "Runs-management/kaggle_validation_6arms_20261104/02_kaggle_qwen_q4_b2_rand.ipynb",\n      "file_sha256_at_audit": "ccef18aeb640a0ded8d5050e54e58a6e504cd0986460288425fba7a5c7dfed28",\n      "configured_inference_seed": 4294967295,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "Runs-management/kaggle_validation_6arms_20261104/02_kaggle_qwen_q4_b2_rep_validation.ipynb",\n      "file_sha256_at_audit": "2e6bfb53c9fc4525a014a9d79ad862b5c3fbc40fd3bcf8e24729ab91257e7a8e",\n      "configured_inference_seed": 4294967295,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "shared_json_integrated": false\n    }\n  ],\n  "resolved_decisions": [\n    {\n      "id": "build_artifact",\n      "question": "Pin a shared container/build artifact or allow platform builds with provenance? All notebook builders leave exact CUDA/compiler/CMake/Python patch unpinned.",\n      "status": "resolved",\n      "decision": "Kaggle platform build with mandatory full provenance"\n    },\n    {\n      "id": "cpu_threads",\n      "question": "Keep CPU-count default formulas or lock absolute thread counts? All 16 constructors omit n_threads and n_threads_batch.",\n      "status": "resolved",\n      "interim_policy": "preserve existing formulas and record effective values",\n      "decision": "preserve default CPU-count formulas and record effective counts"\n    }\n  ],\n  "integrity": {\n    "algorithm": "sha256",\n    "canonicalization": "UTF-8 json.dumps(sort_keys=True,separators=(comma,colon),ensure_ascii=False); exclude integrity",\n    "sha256": "c0f17d3562541fa97e92b46413fa4eb56d2823c59cc9c1bf920a4c404ad3a171"\n  }\n}\n'
RUNTIME_LOCK_PATH = Path('/kaggle/working/inference_runtime_lock.json')
if RUNTIME_LOCK_PATH.exists() and RUNTIME_LOCK_PATH.read_bytes() != RUNTIME_LOCK_BYTES:
    raise ValueError('Working directory has a different runtime lock; start a fresh Kaggle session')
RUNTIME_LOCK_PATH.write_bytes(RUNTIME_LOCK_BYTES)
Path('/kaggle/working/grace_runtime_contract.py').write_text("import contextlib\nimport hashlib\nimport importlib.metadata\nimport json\nimport multiprocessing\nimport os\nimport platform\nimport re\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nEXPECTED_CONTRACT_SHA256 = '85d8770839f80260cdc7c27b27b6cce6bf8815675c8593db68fb102589c5cbd6'\nCONTRACT_PATH = Path('/kaggle/working/inference_runtime_lock.json')\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open('rb') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b''): h.update(block)\n    return h.hexdigest()\n\ndef load_contract(path=CONTRACT_PATH):\n    raw = Path(path).read_bytes()\n    if hashlib.sha256(raw).hexdigest() != EXPECTED_CONTRACT_SHA256:\n        raise ValueError('Runtime contract file SHA-256 mismatch')\n    c = json.loads(raw)\n    unsigned = {k:v for k,v in c.items() if k != 'integrity'}\n    seal = hashlib.sha256(json.dumps(unsigned,sort_keys=True,separators=(',',':'),ensure_ascii=False).encode()).hexdigest()\n    if seal != c['integrity']['sha256']:\n        raise ValueError('Runtime contract canonical integrity mismatch')\n    return c\n\ndef command(args):\n    p = subprocess.run(args, check=True, text=True, capture_output=True)\n    return p.stdout + p.stderr\n\ndef assert_runtime(c):\n    actual = {name:importlib.metadata.version(name) for name in c['packages']}\n    if actual != c['packages']: raise RuntimeError(f'Package mismatch: {actual}')\n    if platform.system() != 'Linux' or '.'.join(map(str,sys.version_info[:2])) != c['build']['python_family']:\n        raise RuntimeError('Contract requires Kaggle Linux/Python 3.13')\n    gpu = command(['nvidia-smi','--query-gpu=index,uuid,name,memory.free,memory.total,driver_version','--format=csv,noheader'])\n    lines = [s for s in gpu.splitlines() if s.strip()]\n    if len(lines) != c['gpu']['device_count'] or any('T4' not in s for s in lines):\n        raise RuntimeError(f'Contract requires 2 T4 GPUs: {gpu}')\n    dist = importlib.metadata.distribution('llama-cpp-python')\n    origin = json.loads(dist.read_text('direct_url.json') or '{}')\n    if origin.get('archive_info',{}).get('hashes',{}).get('sha256') != c['backend_source']['sdist_sha256']:\n        raise RuntimeError('Backend installation is not from the verified source archive')\n    return actual, gpu\n\ndef provenance(c, runner, target):\n    packages, gpu = assert_runtime(c)\n    dist = importlib.metadata.distribution('llama-cpp-python')\n    binaries = {}\n    for entry in dist.files or []:\n        if str(entry).endswith(('.so','.dll','.dylib')):\n            p = Path(dist.locate_file(entry)); binaries[str(entry)] = digest(p)\n    if not binaries: raise RuntimeError('No compiled backend library found for provenance')\n    result = dict(runtime_contract_sha256=EXPECTED_CONTRACT_SHA256,\n        runtime_contract_integrity=c['integrity']['sha256'], packages=packages,\n        python=sys.version, platform=platform.platform(), libc=platform.libc_ver(),\n        gpu_before_load=gpu, cpu_count=multiprocessing.cpu_count(),\n        n_threads=max(multiprocessing.cpu_count()//2,1), n_threads_batch=multiprocessing.cpu_count(),\n        constructor=c['constructor'], runner_sha256=digest(runner),\n        cuda_device_order=os.environ.get('CUDA_DEVICE_ORDER'),\n        cuda_visible_devices=os.environ.get('CUDA_VISIBLE_DEVICES'),\n        backend_sdist_sha256=c['backend_source']['sdist_sha256'], compiled_backend_sha256=binaries,\n        build_environment={k:os.environ.get(k) for k in ('CMAKE_ARGS','CMAKE_BUILD_PARALLEL_LEVEL','FORCE_CMAKE')})\n    for name,args in [('nvcc',['nvcc','--version']),('compiler',['cc','--version']),\n                      ('cxx_compiler',['c++','--version']),('cmake',['cmake','--version']),\n                      ('pip_freeze',[sys.executable,'-m','pip','freeze','--all'])]:\n        result[name] = command(args)\n    target = Path(target); target.parent.mkdir(parents=True,exist_ok=True)\n    target.write_text(json.dumps(result,indent=2),encoding='utf-8')\n    return result\n\ndef load_locked_model(c, model_path, tokenizer_manifest, gpu_layers, log_path, model_digest):\n    import llama_cpp\n    if model_digest != c['model']['sha256']: raise ValueError('Frozen model SHA-256 mismatch')\n    if gpu_layers != c['constructor']['n_gpu_layers']: raise ValueError('GPU layers differ from lock')\n    if (tokenizer_manifest['tokenizer'],tokenizer_manifest['tokenizer_revision'],tokenizer_manifest['prompt_budget_tokens']) != (\n        c['tokenizer']['repo'],c['tokenizer']['revision'],c['constructor']['n_ctx']):\n        raise ValueError('Tokenizer/context differ from runtime lock')\n    if not llama_cpp.llama_supports_gpu_offload(): raise RuntimeError('CUDA backend unavailable')\n    kwargs = dict(c['constructor'])\n    kwargs['split_mode'] = getattr(llama_cpp,kwargs['split_mode'])\n    kwargs.update(n_threads=max(multiprocessing.cpu_count()//2,1), n_threads_batch=multiprocessing.cpu_count())\n    # The pinned Python binding's logger writes through sys.stderr, so capture that stream directly.\n    path = Path(log_path)\n    with path.open('w',encoding='utf-8') as log, contextlib.redirect_stderr(log):\n        model = llama_cpp.Llama(model_path=str(model_path),verbose=True,**kwargs)\n    text = path.read_text(encoding='utf-8',errors='replace')\n    print(text,flush=True)\n    reports = re.findall(r'offloaded\\s+(\\d+)\\s*/\\s*(\\d+)\\s+layers?\\s+to\\s+GPU',text,re.I)\n    required = c['gpu']['required_offloaded_layers']\n    if not any(int(a)==int(b)==required for a,b in reports):\n        model.close()\n        raise RuntimeError(f'Full {required}/{required} GPU offload not verified; inspect {path}')\n    for field in ('n_ctx', 'n_batch', 'n_ubatch'):\n        observed = re.findall(rf'\\b{field}\\s*=\\s*(\\d+)\\b', text)\n        if not observed or int(observed[-1]) != c['constructor'][field]:\n            model.close()\n            raise RuntimeError(f'Effective {field} differs from contract; inspect {path}')\n    gpu_devices = re.findall(r'Device\\s+\\d+:\\s+[^\\n]*\\bT4\\b', text)\n    if len(gpu_devices) != c['gpu']['device_count']:\n        model.close()\n        raise RuntimeError(f'Expected two visible T4 devices in backend log; inspect {path}')\n    return model\n", encoding='utf-8')
if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')
from grace_runtime_contract import assert_runtime, digest, load_contract, EXPECTED_CONTRACT_SHA256
RUNTIME = load_contract()
shutil.copy2(RUNTIME_LOCK_PATH, DIAG / RUNTIME_LOCK_PATH.name)


In [ ]:
import math
ARM = 'B2-rep'
KAGGLE_INPUT_ROOT = Path('/kaggle/input')
BUNDLE_WORK = Path('/kaggle/working/b2_rep_test_bundles')
OUT_ROOT = Path('/kaggle/working/b2_rep_test_runtime_locked')
BUNDLE_NAME = 'grace_b2_rep_test_inputs.zip'
BUNDLE_SHA256 = 'a58dea213a076c07da228a356ad7c754a3fb599a4dbbc765254999218ca3ebfa'
CHECKPOINT_NAME = 'grace_b2_rep_test_runtime_locked_checkpoint.zip'
CHECKPOINT_INPUT_PATH = None
MODEL_FILENAME = 'Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf'
MODEL_SHA256 = 'f6050b4da22869aa3fb9d94bb3314e9fc5a027d4741c1f97761c387d6b2b2335'
MAX_RUNTIME_HOURS = 9
CHECKPOINT_RESERVE_SECONDS = 300
GPU_LAYERS = -1
SETS = [{"name": "seed_42/devign_test.zip", "sha256": "08cced6e5d932cc3e9d727d69e2e72c0a5f5ac69f1ab8480fa2dca996c483186", "seed": 42, "dataset": "devign"}, {"name": "seed_42/reveal_test.zip", "sha256": "ad8bc722a80fa06afc27b4c064dee5d4f9b24630812113ee83356bc70353d3d2", "seed": 42, "dataset": "reveal"}]
LOCKED_THRESHOLDS = {"devign": 0.7, "reveal": 0.12}

FROZEN_MANIFEST_SHA256 = {(42, 'devign'): '9e7cd8c212323fab0c51433b605ff4515eda1d88d17c07d8e1cbcdeddf78d716', (42, 'reveal'): '2b5af90cbf2c55be6c00d12f58981eb47a49c24161adb3fbb07acfcaffe8fa9e'}

if (MODEL_FILENAME, MODEL_SHA256, GPU_LAYERS) != (
    RUNTIME['model']['filename'], RUNTIME['model']['sha256'], RUNTIME['constructor']['n_gpu_layers']
):
    raise ValueError('Notebook model or GPU settings differ from runtime lock')


In [ ]:
def hash_bytes(data):
    return hashlib.sha256(data).hexdigest()

BUNDLE_WORK.mkdir(parents=True, exist_ok=True)
OUT_ROOT.mkdir(parents=True, exist_ok=True)
bundles = {}
expected = {}

# Check whether input bundle is available as an archive file or auto-unpacked by Kaggle
bundle_candidates = sorted(KAGGLE_INPUT_ROOT.rglob(BUNDLE_NAME))
if not bundle_candidates:
    bundle_candidates = sorted(KAGGLE_INPUT_ROOT.rglob(BUNDLE_NAME + '.*'))

if bundle_candidates and len(bundle_candidates) == 1:
    bundle_path = bundle_candidates[0]
    if hash_bytes(bundle_path.read_bytes()) != BUNDLE_SHA256:
        raise ValueError(f'Unexpected input bundle SHA-256 for {bundle_path}')
    note(f'Loaded packed bundle: {bundle_path} (verified SHA-256 {BUNDLE_SHA256})')
    with zipfile.ZipFile(bundle_path) as bundle:
        if sorted(bundle.namelist()) != sorted(s['name'] for s in SETS):
            raise ValueError('Unexpected bundle members')
        for spec in SETS:
            seed, ds = spec['seed'], spec['dataset']
            payload = bundle.read(spec['name'])
            if hash_bytes(payload) != spec['sha256']:
                raise ValueError(f'SHA-256 mismatch for member {spec["name"]}')
            path = BUNDLE_WORK / f'{seed}_{ds}.zip'
            path.write_bytes(payload)
            bundles[seed, ds] = path
            with zipfile.ZipFile(path) as inner:
                records = [json.loads(line) for line in inner.read('prompts.jsonl').splitlines()]
            expected[seed, ds] = {r['id']: (r['prompt_sha256'], r['label']) for r in records}
else:
    note(f'Bundle archive {BUNDLE_NAME} not found as single file; discovering Kaggle auto-unpacked files under {KAGGLE_INPUT_ROOT}')
    for spec in SETS:
        seed, ds = spec['seed'], spec['dataset']
        manifest_candidates = sorted(p for p in KAGGLE_INPUT_ROOT.rglob('bundle_manifest.json')
                                     if f'seed_{seed}' in str(p).replace('\\', '/') and ds in str(p).replace('\\', '/'))
        if len(manifest_candidates) != 1:
            raise FileNotFoundError(f'Expected 1 bundle_manifest.json for seed {seed} {ds}; found {manifest_candidates}')
        manifest_file = manifest_candidates[0]
        prompts_file = manifest_file.parent / 'prompts.jsonl'
        if not prompts_file.is_file():
            raise FileNotFoundError(f'Missing prompts.jsonl next to {manifest_file}')
        manifest_bytes = manifest_file.read_bytes()
        if hash_bytes(manifest_bytes) != FROZEN_MANIFEST_SHA256[seed, ds]:
            raise ValueError(f'Frozen manifest SHA-256 mismatch: {manifest_file}')
        manifest = json.loads(manifest_bytes)
        prompts_bytes = prompts_file.read_bytes()
        if hash_bytes(prompts_bytes) != manifest['prompts_sha256']:
            raise ValueError(f'Prompt SHA256 mismatch for {prompts_file}: expected {manifest["prompts_sha256"]}')
        if (manifest.get('arm'), manifest.get('seed'), manifest.get('dataset'), manifest.get('split')) != (ARM, seed, ds, 'test'):
            raise ValueError(f'Manifest identity mismatch in {manifest_file}: {manifest}')
        path = BUNDLE_WORK / f'{seed}_{ds}.zip'
        with zipfile.ZipFile(path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
            z.writestr('bundle_manifest.json', manifest_bytes)
            z.writestr('prompts.jsonl', prompts_bytes)
        bundles[seed, ds] = path
        records = [json.loads(line) for line in prompts_bytes.decode('utf-8').splitlines() if line.strip()]
        expected[seed, ds] = {r['id']: (r['prompt_sha256'], r['label']) for r in records}
        note(f'Loaded auto-unpacked bundle for seed {seed} {ds}: {len(expected[seed, ds])} prompts, verified SHA-256')

allowed = {f'seed_{seed}/{ds}/{split}/{file}'
           for seed, ds in bundles for file in ('records.jsonl', 'run_manifest.json') for split in ('test', 'validation')}
if CHECKPOINT_INPUT_PATH is not None:
    checkpoint = Path(CHECKPOINT_INPUT_PATH)
else:
    candidates = sorted(KAGGLE_INPUT_ROOT.rglob(CHECKPOINT_NAME))
    if len(candidates) > 1:
        raise ValueError('Set CHECKPOINT_INPUT_PATH to the intended checkpoint')
    checkpoint = candidates[0] if candidates else None

if checkpoint:
    if any((OUT_ROOT / name).exists() for name in allowed):
        raise ValueError('Output already contains records. Clear CHECKPOINT_INPUT_PATH and detach the checkpoint to continue local output.')
    with zipfile.ZipFile(checkpoint) as source:
        if source.namelist().count(RUNTIME_LOCK_PATH.name) != 1 or source.read(RUNTIME_LOCK_PATH.name) != RUNTIME_LOCK_BYTES:
            raise ValueError('Checkpoint was created under a missing or different runtime lock')
        names = [n for n in source.namelist() if n in allowed or re.fullmatch(
            r'runtime_provenance/seed_\d+_[a-z0-9_]+/(runtime_provenance.json|model_load_offload.log)', n)]
        if any(n.endswith('records.jsonl') and n.replace('records.jsonl', 'run_manifest.json') not in names
               for n in names):
            raise ValueError('Checkpoint is missing a run manifest')
        if len(names) != len(set(names)):
            raise ValueError('Duplicate checkpoint entries')
        if not any(n.endswith('records.jsonl') for n in names):
            raise ValueError('Checkpoint has no records for this run')
        for name in names:
            target = OUT_ROOT / name
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(source.read(name))

shutil.copy2(RUNTIME_LOCK_PATH, OUT_ROOT / RUNTIME_LOCK_PATH.name)

def validate_records():
    progress = {}
    for (seed, ds), prompts in expected.items():
        folder = OUT_ROOT / f'seed_{seed}' / ds / 'test'
        path = folder / 'records.jsonl'
        seen = set()
        if path.exists():
            run_manifest = json.loads((folder / 'run_manifest.json').read_text(encoding='utf-8'))
            if any(run_manifest.get(key) != value for key, value in {
                'runtime_contract_sha256': EXPECTED_CONTRACT_SHA256, 'arm': ARM, 'prompt_seed': seed,
                'dataset': ds, 'split': 'test', 'model_sha256': MODEL_SHA256,
                'tokenizer_revision': RUNTIME['tokenizer']['revision'],
                'scoring': RUNTIME['scoring']['name'],
                'backend_sdist_sha256': RUNTIME['backend_source']['sdist_sha256'],
                'constructor': RUNTIME['constructor'],
            }.items()):
                raise ValueError(f'Checkpoint run identity differs from runtime lock: {path}')
            for line in path.read_text(encoding='utf-8').splitlines():
                row = json.loads(line)
                uid = row['id']
                identity = (row['arm'], row['seed'], row['dataset'], row['split'], row['model_sha256'])
                if (uid in seen or identity != (ARM, seed, ds, 'test', MODEL_SHA256)
                        or prompts.get(uid) != (row['prompt_sha256'], row['label'])
                        or row.get('runtime_contract_sha256') != EXPECTED_CONTRACT_SHA256
                        or row.get('inference_seed') != RUNTIME['constructor']['seed']
                        or (ARM == 'B2-rep' and (row.get('repeat') is not True or row.get('tag') != 'repeat'))
                        or not math.isfinite(row['s'])):
                    raise ValueError(f'Unexpected record: {path}/{uid}')
                seen.add(uid)
        progress[seed, ds] = len(seen)
    return progress

In [ ]:
# Install the approved versions, then build the verified source archive.
import urllib.request
build = RUNTIME['build']
os.environ['CMAKE_ARGS'] = (
    f"-DGGML_CUDA={'on' if build['GGML_CUDA'] else 'off'}"
    f" -DCMAKE_CUDA_ARCHITECTURES={build['CMAKE_CUDA_ARCHITECTURES']}"
    f" -DCMAKE_LIBRARY_PATH={build['CMAKE_LIBRARY_PATH']}"
)
for key in ('CMAKE_BUILD_PARALLEL_LEVEL', 'FORCE_CMAKE'):
    os.environ[key] = build[key]
packages = [f'{name}=={version}' for name, version in RUNTIME['packages'].items()
            if name != RUNTIME['backend_source']['name']]
packages += ['typing-extensions>=4.5.0', 'diskcache>=5.6.1', 'jinja2>=2.11.3']
run_logged([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', *packages], 'install_dependencies')
source = RUNTIME['backend_source']
with urllib.request.urlopen(f"https://pypi.org/pypi/{source['name']}/{source['version']}/json", timeout=60) as response:
    metadata = json.load(response)
artifact = next(item for item in metadata['urls'] if item['filename'] == source['sdist_filename'])
if artifact['digests']['sha256'] != source['sdist_sha256']:
    raise ValueError('Backend source registry SHA-256 mismatch')
archive_path = Path('/kaggle/working') / source['sdist_filename']
with urllib.request.urlopen(artifact['url'], timeout=120) as response, archive_path.open('wb') as target:
    shutil.copyfileobj(response, target)
if digest(archive_path) != source['sdist_sha256']:
    raise ValueError('Backend source file SHA-256 mismatch')
run_logged([sys.executable, '-m', 'pip', 'install', '-v', '--no-cache-dir', '--no-deps',
            '--force-reinstall', str(archive_path)], 'install_llama_cuda')
note('Locked packages: ' + json.dumps(assert_runtime(RUNTIME)[0], sort_keys=True))
diagnostics_zip()


In [ ]:
assert_runtime(RUNTIME)
import llama_cpp
versions()
note('llama-cpp-python GPU offload: ' + str(llama_cpp.llama_supports_gpu_offload()))
assert llama_cpp.llama_supports_gpu_offload(), 'llama-cpp-python chưa có CUDA; xem install_llama_cuda.log'
diagnostics_zip()


In [ ]:
# Tìm đúng tên file model trong toàn bộ Input đã gắn, bất kể username/slug.
model_files = sorted(path for path in KAGGLE_INPUT_ROOT.rglob(MODEL_FILENAME) if path.is_file())
if len(model_files) != 1:
    raise FileNotFoundError(f'Cần đúng một {MODEL_FILENAME} trong Kaggle Input; '
                            f'tìm thấy: {list(map(str, model_files))}. '
                            'Hãy Add Input chứa model và bỏ bản trùng nếu có; '
                            'notebook không tự tải lại GGUF từ Hugging Face.')
model_path = str(model_files[0])
revision = 'local-input'
note(f'Dùng model input: {model_path}')
print('Model input:', model_path)
print('Batch output directory:', OUT_ROOT)


In [ ]:
%%writefile /kaggle/working/kaggle_inference.py
import argparse
import hashlib
import json
import math
import os
import platform
import time
import tempfile
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
from grace_runtime_contract import (
    EXPECTED_CONTRACT_SHA256, command, digest, load_contract, load_locked_model, provenance,
)

RUNTIME = load_contract()


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def write_json_atomic(path, payload):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    os.replace(tmp, path)


def binary_scores(logits, label_ids):
    values = np.asarray(logits, dtype=np.float64).reshape(-1)
    if not np.all(np.isfinite(values)):
        raise ValueError('Non-finite logits')
    maximum = np.max(values)
    normalizer = float(maximum + np.log(np.exp(values - maximum).sum()))
    lp0 = float(values[label_ids[0]] - normalizer)
    lp1 = float(values[label_ids[1]] - normalizer)
    score = float(np.exp(lp1 - np.logaddexp(lp0, lp1)))
    best = int(np.argmax(values))
    top = np.argsort(values)[-5:][::-1]
    return {'logprob_0': lp0, 'logprob_1': lp1, 'p0': math.exp(lp0), 'p1': math.exp(lp1),
            's': score, 'pred_argmax': int(score >= 0.5), 'greedy_token_id': best,
            'top5': [{'token_id': int(i), 'logprob': float(values[i] - normalizer)} for i in top]}


def load_bundle(bundle):
    with zipfile.ZipFile(bundle) as z:
        manifest = json.loads(z.read('bundle_manifest.json'))
        prompt_bytes = z.read('prompts.jsonl')
    if hashlib.sha256(prompt_bytes).hexdigest() != manifest['prompts_sha256']:
        raise ValueError(f'Prompt bundle hash mismatch: {bundle}')
    rows = [json.loads(line) for line in prompt_bytes.decode('utf-8').splitlines() if line.strip()]
    if len(rows) != manifest['count'] or len({str(r['id']) for r in rows}) != len(rows):
        raise ValueError(f'Prompt count or unique ID check failed: {bundle}')
    return manifest, rows


def evaluate_output(output, tau_star=None):
    from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                                 confusion_matrix, f1_score, matthews_corrcoef, precision_score,
                                 recall_score, roc_auc_score)
    records = [json.loads(line) for line in output.read_text(encoding='utf-8').splitlines() if line.strip()]
    actual = [r for r in records if r.get('response_valid', True)]
    truth = [r['label'] for r in actual]
    predicted = [int(r['raw']) for r in actual if str(r.get('raw', '')).strip() in ('0', '1')]
    metrics = {'n_total': len(records), 'n_score_valid': sum(bool(r['valid']) for r in records),
               'n_response_valid': len(actual),
               'response_coverage': len(actual) / len(records) if records else None,
               'accuracy': None, 'balanced_accuracy': None, 'precision': None, 'recall': None,
               'f1': None, 'mcc': None, 'confusion_matrix': None, 'auroc': None, 'auprc': None}
    if actual and len(predicted) == len(truth):
        metrics.update(accuracy=float(accuracy_score(truth, predicted)),
                       balanced_accuracy=float(balanced_accuracy_score(truth, predicted)),
                       precision=float(precision_score(truth, predicted, zero_division=0)),
                       recall=float(recall_score(truth, predicted, zero_division=0)),
                       f1=float(f1_score(truth, predicted, zero_division=0)),
                       mcc=float(matthews_corrcoef(truth, predicted)),
                       confusion_matrix=confusion_matrix(truth, predicted, labels=[0, 1]).tolist())
    score_rows = [r for r in records if r['valid']]
    if score_rows and len({r['label'] for r in score_rows}) == 2:
        labels = [r['label'] for r in score_rows]
        scores = [r['s'] for r in score_rows]
        metrics.update(auroc=float(roc_auc_score(labels, scores)),
                       auprc=float(average_precision_score(labels, scores)))
    if tau_star is not None and score_rows:
        y_true = np.array([r['label'] for r in score_rows])
        y_score = np.array([r['s'] for r in score_rows])
        y_pred = (y_score >= tau_star).astype(int)
        metrics.update(
            tau_star=float(tau_star),
            mcc_at_tau=float(matthews_corrcoef(y_true, y_pred)),
            ba_at_tau=float(balanced_accuracy_score(y_true, y_pred)),
            f1_at_tau=float(f1_score(y_true, y_pred, zero_division=0))
        )
    return metrics

    from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                                 confusion_matrix, f1_score, matthews_corrcoef, precision_score,
                                 recall_score, roc_auc_score)
    records = [json.loads(line) for line in output.read_text(encoding='utf-8').splitlines() if line.strip()]
    actual = [r for r in records if r['response_valid']]
    truth = [r['label'] for r in actual]
    predicted = [int(r['raw']) for r in actual]
    metrics = {'n_total': len(records), 'n_score_valid': sum(bool(r['valid']) for r in records),
               'n_response_valid': len(actual),
               'response_coverage': len(actual) / len(records) if records else None,
               'accuracy': None, 'balanced_accuracy': None, 'precision': None, 'recall': None,
               'f1': None, 'mcc': None, 'confusion_matrix': None, 'auroc': None, 'auprc': None}
    if actual:
        metrics.update(accuracy=accuracy_score(truth, predicted),
                       balanced_accuracy=balanced_accuracy_score(truth, predicted),
                       precision=precision_score(truth, predicted, zero_division=0),
                       recall=recall_score(truth, predicted, zero_division=0),
                       f1=f1_score(truth, predicted, zero_division=0),
                       mcc=matthews_corrcoef(truth, predicted),
                       confusion_matrix=confusion_matrix(truth, predicted, labels=[0, 1]).tolist())
    score_rows = [r for r in records if r['valid']]
    if score_rows and len({r['label'] for r in score_rows}) == 2:
        labels = [r['label'] for r in score_rows]
        scores = [r['s'] for r in score_rows]
        metrics.update(auroc=roc_auc_score(labels, scores),
                       auprc=average_precision_score(labels, scores))
    return metrics


def run_batch(runs, model_path, model_revision, gpu_layers, limit, deadline_epoch,
              checkpoint_reserve_seconds, status_path, seed, expected_model_sha256, arm):
    if arm != 'B2-rep':
        raise ValueError('This worker is reserved for B2-rep')
    from llama_cpp import __version__ as llama_version
    from transformers import AutoTokenizer
    if llama_version != RUNTIME['backend_source']['version']: raise ValueError(f'Backend version mismatch: {llama_version}')

    started = time.time()
    stop_at = deadline_epoch - checkpoint_reserve_seconds
    status_path = Path(status_path)
    status_path.parent.mkdir(parents=True, exist_ok=True)
    status = {'state': 'starting', 'timed_out': False,
              'started_at': datetime.fromtimestamp(started, timezone.utc).isoformat(),
              'deadline_at': datetime.fromtimestamp(deadline_epoch, timezone.utc).isoformat(),
              'stop_new_records_at': datetime.fromtimestamp(stop_at, timezone.utc).isoformat(),
              'max_runtime_hours': (deadline_epoch - started) / 3600,
              'checkpoint_reserve_seconds': checkpoint_reserve_seconds,
              'runs': {}, 'completed_records_total': 0}
    for dataset, split, bundle, out in runs:
        key = f'{dataset}/{split}'
        status['runs'][key] = {'state': 'pending', 'bundle': bundle, 'output': out}
    write_json_atomic(status_path, status)

    def stop_for_deadline(current=None):
        status['state'] = 'timed_out'
        status['timed_out'] = True
        status['timed_out_at'] = datetime.now(timezone.utc).isoformat()
        status['stopped_after_run'] = current
        write_json_atomic(status_path, status)

    if time.time() >= stop_at:
        stop_for_deadline(None)
        return

    model_path = Path(model_path)
    model_digest = sha256(model_path)
    if model_digest != expected_model_sha256: raise ValueError('Model SHA256 không khớp contract')
    if model_revision == 'local-input':
        model_revision = f'local-sha256:{model_digest}'
    first_manifest, _ = load_bundle(runs[0][2])
    for dataset, split, bundle, out in runs:
        current_manifest, _ = load_bundle(bundle)
        if (current_manifest['tokenizer'], current_manifest['tokenizer_revision'],
            current_manifest['prompt_budget_tokens']) != (first_manifest['tokenizer'],
            first_manifest['tokenizer_revision'], first_manifest['prompt_budget_tokens']):
            raise ValueError('All four bundles must use the same tokenizer revision and context budget')
    tokenizer = AutoTokenizer.from_pretrained(first_manifest['tokenizer'],
                                                revision=first_manifest['tokenizer_revision'])
    provenance_root = status_path.parent / 'runtime_provenance'
    provenance_root.mkdir(exist_ok=True)
    session = Path(tempfile.mkdtemp(prefix=f'seed_{seed}_', dir=provenance_root))
    provenance_file = session / 'runtime_provenance.json'
    offload_log = session / 'model_load_offload.log'
    runtime_info = provenance(RUNTIME, __file__, provenance_file)
    build_log = Path('/kaggle/working/grace_diagnostics/install_llama_cuda.log')
    runtime_info['backend_build_log_sha256'] = digest(build_log)
    build_lines = build_log.read_text(encoding='utf-8', errors='replace').splitlines()
    runtime_info['backend_build_report'] = [line.strip() for line in build_lines if any(
        key in line for key in ('using CMake', 'compiler identification', 'Created wheel for llama-cpp-python'))]
    llm = load_locked_model(RUNTIME, model_path, first_manifest, gpu_layers, offload_log, model_digest)
    runtime_info['gpu_after_load'] = command([
        'nvidia-smi', '--query-gpu=index,uuid,name,memory.free,memory.total,driver_version',
        '--format=csv,noheader',
    ])
    runtime_info['offload_log_sha256'] = digest(offload_log)
    write_json_atomic(provenance_file, runtime_info)
    provenance_ref = provenance_file.relative_to(status_path.parent).as_posix()
    if time.time() >= stop_at:
        stop_for_deadline(None)
        return
    label_ids = []
    for label in ('0', '1'):
        hf_ids = tokenizer.encode(label, add_special_tokens=False)
        gguf_ids = llm.tokenize(label.encode(), add_bos=False, special=False)
        if len(hf_ids) != 1 or hf_ids != gguf_ids:
            raise ValueError(f'Label token mismatch for {label}: HF={hf_ids}, GGUF={gguf_ids}')
        label_ids.append(hf_ids[0])
    if label_ids[0] == label_ids[1]:
        raise ValueError('Label token IDs must differ')

    status['state'] = 'running'
    write_json_atomic(status_path, status)
    for dataset, split, bundle, out_text in runs:
        key = f'{dataset}/{split}'
        out = Path(out_text)
        out.mkdir(parents=True, exist_ok=True)
        manifest, rows = load_bundle(bundle)
        if (manifest.get('arm'), manifest.get('seed'), manifest['dataset'], manifest['split']) != (arm, seed, dataset, split):
            raise ValueError(f'Bundle identity mismatch for {key}: {manifest["dataset"]}/{manifest["split"]}')
        if limit is not None:
            rows = rows[:limit]
        identity = {'bundle_sha256': sha256(bundle),
                    'prompt_set_sha256': manifest['prompts_sha256'], 'model_sha256': model_digest,
                    'model_repo': manifest['model_repo'], 'quantization': manifest['quantization'],
                    'model_file': model_path.name, 'llama_cpp_python': llama_version,
                    'model_repo_revision': model_revision, 'tokenizer': manifest['tokenizer'],
                    'tokenizer_revision': manifest['tokenizer_revision'], 'gpu_layers': gpu_layers,
                    'context_tokens': manifest['prompt_budget_tokens'],
                    'scoring': RUNTIME['scoring']['name'], 'limit': limit,
                    'runtime_contract_sha256': EXPECTED_CONTRACT_SHA256,
                    'constructor': RUNTIME['constructor'],
                    'backend_sdist_sha256': RUNTIME['backend_source']['sdist_sha256'],
                    'python': platform.python_version(), 'dataset': dataset, 'split': split, 'arm': arm, 'prompt_seed': seed,
                    'repeat': True, 'tag': 'repeat'}
        identity_file = out / 'run_manifest.json'
        previous_identity = (json.loads(identity_file.read_text(encoding='utf-8'))
                             if identity_file.exists() else None)
        if previous_identity is not None:
            # Ignore only archive-byte hashes: ZIP timestamps/compression metadata may change.
            # Prompt identity is verified below against every checkpoint row's prompt_sha256.
            ignored = {'bundle_sha256', 'prompt_set_sha256', 'python'}  # Exact Kaggle build is recorded per session.
            previous_stable = {k: v for k, v in previous_identity.items() if k not in ignored}
            current_stable = {k: v for k, v in identity.items() if k not in ignored}
            if previous_stable != current_stable:
                raise ValueError(f'Existing output has different model/run settings for {key}; use a new OUT_ROOT')
            if ('prompt_set_sha256' in previous_identity and
                    previous_identity['prompt_set_sha256'] != identity['prompt_set_sha256']):
                raise ValueError(f'Prompt bundle content changed for {key}; refusing to resume')
        output = out / 'records.jsonl'
        done = {}
        if output.exists():
            for line in output.read_text(encoding='utf-8').splitlines():
                if line.strip():
                    row = json.loads(line)
                    row_id = str(row['id'])
                    if row_id in done:
                        raise ValueError(f'Duplicate checkpoint ID in {key}: {row_id}')
                    done[row_id] = row
        expected_ids = {str(item['id']) for item in rows}
        unexpected_ids = set(done) - expected_ids
        if unexpected_ids:
            raise ValueError(f'Checkpoint contains IDs absent from current bundle {key}: {sorted(unexpected_ids)[:10]}')
        for item in rows:
            row_id = str(item['id'])
            if row_id in done:
                if done[row_id].get('prompt_sha256') != item['prompt_sha256']:
                    raise ValueError(f'Checkpoint prompt mismatch: {key}/{row_id}')
                if done[row_id].get('repeat') is not True or done[row_id].get('tag') != 'repeat':
                    raise ValueError(f'Checkpoint repeat identity mismatch: {key}/{row_id}')
                if done[row_id].get('model_sha256') != model_digest:
                    raise ValueError(f'Checkpoint model hash mismatch: {key}/{row_id}')
        # Persist the refreshed archive hash only after checkpoint IDs and prompt hashes pass.
        identity_file.write_text(json.dumps(identity, indent=2), encoding='utf-8')
        progress = status['runs'][key]
        progress.update(state='running', count=len(rows), completed=len(done),
                        records_path=str(output), bundle_sha256=identity['bundle_sha256'])
        status['completed_records_total'] = sum(x.get('completed', 0) for x in status['runs'].values())
        write_json_atomic(status_path, status)
        if len(done) > len(rows):
            raise ValueError(f'Checkpoint has more rows than this bundle: {key}')
        print(f'{key}: verified checkpoint IDs={len(done)}; skipping these; remaining={len(rows) - len(done)}', flush=True)

        with output.open('a', encoding='utf-8', newline='\n') as f:
            for item in rows:
                row_id = str(item['id'])
                if row_id in done:
                    if done[row_id]['prompt_sha256'] != item['prompt_sha256']:
                        raise ValueError(f'Checkpoint prompt mismatch: {key}/{row_id}')
                    continue
                if time.time() >= stop_at:
                    progress.update(state='checkpointed', completed=len(done))
                    status['completed_records_total'] = sum(x.get('completed', 0) for x in status['runs'].values())
                    stop_for_deadline(key)
                    return
                prompt = item['prompt']
                if hashlib.sha256(prompt.encode()).hexdigest() != item['prompt_sha256']:
                    raise ValueError(f'Prompt hash mismatch: {key}/{row_id}')
                messages = [{'role': 'user', 'content': prompt}]
                chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                hf_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=False)
                gguf_ids = llm.tokenize(chat.encode('utf-8'), add_bos=False, special=True)
                if list(hf_ids) != gguf_ids:
                    raise ValueError(f'Tokenizer mismatch at {key}/{row_id}; abort rather than alter prompt')
                if len(gguf_ids) > manifest['prompt_budget_tokens']:
                    raise ValueError(f'Token budget exceeded at {key}/{row_id}')
                start = time.monotonic()
                llm.reset()
                llm.eval(gguf_ids)
                scores = binary_scores(llm.scores[llm.n_tokens - 1], label_ids)
                raw = llm.detokenize([scores['greedy_token_id']]).decode('utf-8', errors='replace').strip()
                row = {k: item[k] for k in ('id', 'dataset', 'split', 'label', 'prompt_sha256', 'demo_ids', 'tokens', 'graph_missing')}
                row.update(scores, arm=arm, repeat=True, tag='repeat', raw=raw, response_valid=raw in ('0', '1'), valid=True,
                           token_ids={'0': label_ids[0], '1': label_ids[1]}, model_sha256=model_digest,
                           backend='llama-cpp-python', quantization='Q4_K_M', model_repo=manifest['model_repo'],
                           model_repo_revision=model_revision, seed=seed, timestamp=datetime.now(timezone.utc).isoformat(),
                           logprob_distribution='unconstrained_first_token', elapsed_s=time.monotonic()-start)
                row.update(runtime_contract_sha256=EXPECTED_CONTRACT_SHA256,
                           inference_seed=RUNTIME['constructor']['seed'], runtime_provenance=provenance_ref)
                f.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + '\n')
                f.flush()
                os.fsync(f.fileno())
                done[row_id] = row
                progress['completed'] = len(done)
                status['completed_records_total'] = sum(x.get('completed', 0) for x in status['runs'].values())
                status['last_checkpoint_at'] = datetime.now(timezone.utc).isoformat()
                write_json_atomic(status_path, status)
                print(f'{key} {row_id}: s={scores["s"]:.4f}, raw={raw!r} ({len(done)}/{len(rows)})', flush=True)
                if time.time() >= stop_at:
                    progress['state'] = 'checkpointed'
                    stop_for_deadline(key)
                    return

        metrics = evaluate_output(output, tau_star=tau_star)
        (out / 'evaluation.json').write_text(json.dumps(metrics, indent=2, allow_nan=False), encoding='utf-8')
        progress.update(state='complete', completed=len(rows), evaluation=str(out / 'evaluation.json'))
        status['completed_records_total'] = sum(x.get('completed', 0) for x in status['runs'].values())
        write_json_atomic(status_path, status)

    status['state'] = 'complete'
    status['timed_out'] = False
    status['completed_at'] = datetime.now(timezone.utc).isoformat()
    write_json_atomic(status_path, status)


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--model', required=True)
    ap.add_argument('--seed', type=int, required=True)
    ap.add_argument('--arm', required=True)
    ap.add_argument('--expected-model-sha256', required=True)
    ap.add_argument('--model-revision', required=True)
    ap.add_argument('--gpu-layers', type=int, default=-1)
    ap.add_argument('--limit', type=int)
    ap.add_argument('--deadline-epoch', type=float, required=True)
    ap.add_argument('--checkpoint-reserve-seconds', type=int, default=300)
    ap.add_argument('--status', required=True)
    ap.add_argument('--run', action='append', nargs=4, metavar=('DATASET', 'SPLIT', 'BUNDLE', 'OUT'), required=True)
    args = ap.parse_args()
    run_batch(args.run, args.model, args.model_revision, args.gpu_layers, args.limit,
              args.deadline_epoch, args.checkpoint_reserve_seconds, args.status, args.seed, args.expected_model_sha256, args.arm)


In [ ]:
deadline = SESSION_STARTED + MAX_RUNTIME_HOURS * 3600
state = 'error'
progress = validate_records()
try:
    for seed in sorted({seed for seed, ds in bundles}):
        needed = [ds for ds in ('devign', 'reveal')
                  if (seed, ds) in bundles and progress[seed, ds] < len(expected[seed, ds])]
        if not needed:
            continue
        if time.time() >= deadline - CHECKPOINT_RESERVE_SECONDS:
            break
        status = OUT_ROOT / f'seed_{seed}_status.json'
        command = [sys.executable, '-u', '/kaggle/working/kaggle_inference.py',
                   '--model', model_path, '--model-revision', revision,
                   '--seed', str(seed), '--arm', ARM, '--expected-model-sha256', MODEL_SHA256,
                   '--gpu-layers', str(GPU_LAYERS), '--deadline-epoch', str(deadline),
                   '--checkpoint-reserve-seconds', str(CHECKPOINT_RESERVE_SECONDS), '--status', str(status)]
        for ds in needed:
            command += ['--run', ds, 'test', str(bundles[seed, ds]),
                        str(OUT_ROOT / f'seed_{seed}' / ds / 'test')]
        run_logged(command, f'test_seed_{seed}')
        progress = validate_records()
        result = json.loads(status.read_text(encoding='utf-8'))
        if result['timed_out']:
            break
        if any(progress[seed, ds] != len(expected[seed, ds]) for ds in needed):
            raise RuntimeError('Test stopped without a timeout or complete output')
    complete = all(progress[key] == len(expected[key]) for key in expected)
    state = 'test_complete' if complete else 'timed_out'
finally:
    summary = {'runtime_contract_sha256': EXPECTED_CONTRACT_SHA256, 'arm': ARM, 'state': state, 'split': 'test', 'bundle_sha256': BUNDLE_SHA256,
               'progress': {f'{seed}/{ds}': progress[seed, ds] for seed, ds in progress}}
    (OUT_ROOT / 'batch_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    archive = shutil.make_archive(str(Path('/kaggle/working') / CHECKPOINT_NAME.removesuffix('.zip')),
                                  'zip', root_dir=OUT_ROOT)
    print(state, archive)
    diagnostics_zip()
